# Clase 3 · Notebook 02
# Tu primera red de neuronas con Keras (resolviendo el XOR)

**Introducción al Deep Learning — Módulo 2, Unidad 1: Redes de neuronas**

En el Notebook 01 vimos que un perceptrón simple **no** puede resolver el XOR. Ahora construiremos un
**perceptrón multicapa (MLP)** con **Keras** (la API de alto nivel de TensorFlow) y comprobaremos que **sí** lo aprende.

Recorreremos el flujo de una red de neuronas:
1. Preparar los datos.
2. Definir la arquitectura (capas `Dense`, funciones de activación).
3. Compilar (optimizador y función de pérdida).
4. Entrenar (`fit`) y observar la curva de pérdida.
5. Evaluar las predicciones.

> 💡 En Google Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Importar y preparar los datos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
tf.random.set_seed(42); np.random.seed(42)
print("TensorFlow", tf.__version__)

# Las cuatro combinaciones del XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype="float32")
y = np.array([[0], [1], [1], [0]], dtype="float32")
print("Entradas X:\n", X)
print("Salida esperada (XOR):", y.ravel())

## 2. Definir la arquitectura

Un MLP para el XOR necesita al menos una **capa oculta**:

- **Capa oculta**: 8 neuronas con activación **ReLU**.
- **Capa de salida**: 1 neurona con activación **sigmoide** (devuelve una probabilidad entre 0 y 1).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

modelo = Sequential([
    Input(shape=(2,)),
    Dense(8, activation="relu"),       # capa oculta
    Dense(1, activation="sigmoid"),    # capa de salida
])
modelo.summary()

## 3. Compilar el modelo

Elegimos:
- **Optimizador**: `adam` (descenso del gradiente adaptativo, visto en la Clase 2).
- **Función de pérdida**: `binary_crossentropy` (clasificación binaria).
- **Métrica**: exactitud (`accuracy`).

In [ ]:
modelo.compile(optimizer="adam",
               loss="binary_crossentropy",
               metrics=["accuracy"])
print("Modelo compilado.")

## 4. Entrenar el modelo

`fit` ejecuta la **propagación hacia delante**, el cálculo del error y la **propagación hacia atrás**
(backpropagation) durante el número de épocas indicado.

In [ ]:
historia = modelo.fit(X, y, epochs=1000, verbose=0)
print("Entrenamiento terminado.")
print("Pérdida final:    %.4f" % historia.history["loss"][-1])
print("Exactitud final:  %.0f %%" % (historia.history["accuracy"][-1] * 100))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(historia.history["loss"], color="#000F9F", lw=2)
plt.title("Curva de pérdida durante el entrenamiento")
plt.xlabel("Época"); plt.ylabel("Pérdida (loss)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Evaluar las predicciones

Comprobamos que la red ha aprendido el XOR.

In [ ]:
pred = modelo.predict(X, verbose=0)
print("Predicciones del MLP:\n")
print(" x1 x2 | prob.  | predicho | real")
print(" ------+--------+----------+-----")
for xi, pi, yi in zip(X, pred.ravel(), y.ravel()):
    print(f"  {int(xi[0])}  {int(xi[1])} | {pi:.3f}  |    {int(round(pi))}     |  {int(yi)}")
print("\n¡El perceptrón multicapa SÍ resuelve el XOR!")

## Experimenta tú

1. Quita la capa oculta (deja solo la `Dense(1, ...)`) y reentrena. ¿Vuelve a fallar como el perceptrón simple?
2. Cambia el número de neuronas o la activación de la capa oculta (`tanh` en lugar de `relu`).
3. Reduce las épocas a 50. ¿Llega a aprender?

## Resumen

Has construido y entrenado tu primera red de neuronas:

- **Sequential + Dense**: definir capas y sus funciones de activación.
- **compile**: elegir optimizador (`adam`) y pérdida (`binary_crossentropy`).
- **fit**: entrenar (forward + backpropagation).
- **predict / evaluate**: comprobar el resultado.

Una capa oculta basta para resolver un problema no lineal como el XOR. En la próxima clase aplicaremos
este mismo esquema a un problema de **clasificación** real.